In [32]:
from pathlib import Path
import re
from pprint import pprint
from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

In [27]:
# Questions 目录
QUESTIONS_DIR = Path("./Questions")

# ———————————————————— 参数：拆分用 ————————————————————
# 8 个固定 section
SECTION_TITLES = [
    "Problem Summary",
    "Formal Problem Definition",
    "Input Specification",
    "Output Specification",
    "Constraints",
    "Key Observations",
    "Algorithm Idea",
    "Detailed Algorithm Steps",
]

SECTION_VARS = {
    "Problem Summary": "ProblemSummary",
    "Formal Problem Definition": "FormalProblemDefinition",
    "Input Specification": "InputSpecification",
    "Output Specification": "OutputSpecification",
    "Constraints": "Constraints",
    "Key Observations": "KeyObservations",
    "Algorithm Idea": "AlgorithmIdea",
    "Detailed Algorithm Steps": "DetailedAlgorithmSteps",
}

FIRST_SECTION_MARK = "### [1] Problem Summary"

SECTION_PATTERN = re.compile(
    r"^###\s*\[(\d+)\]\s*(.+?)\s*$",
    re.MULTILINE
)

NUMBERED_STEP_RE = re.compile(r"^\s*\d+\.\s+")



# ———————————————————— 参数：CrossEncoder计算用 ————————————————————
model = CrossEncoder("cross-encoder/stsb-roberta-large")

# 块名顺序
KEY_ORDER = [
    "ProblemSummary",               # 1
    "FormalProblemDefinition",      # 2
    "InputSpecification",           # 3
    "OutputSpecification",          # 4
    "Constraints",                  # 5
    "KeyObservations",              # 6
    "AlgorithmIdea",                # 7
    "DetailedAlgorithmSteps",       # 8
    "Note",                         # 9
]

# 对应矩阵变量名
MAT_NAMES = [
    "mat1PS",
    "mat2FPD",
    "mat3IS",
    "mat4OS",
    "mat5C",
    "mat6KO",
    "mat7AI",
    "mat8DAS",
    "mat9N",
]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# 拆分用 函数部分
def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


def list_explain_txts(explain_dir: Path):
    txts = []

    for p in explain_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".txt"):
            continue

        parts = p.stem.split()

        if len(parts) >= 1 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    display(txts_sorted)

    return [p for _, p in txts_sorted]


def remove_blank_lines(text: str) -> str:
    """
    对块内容做空行剔除：删除所有空行，仅保留非空行并按原顺序拼接。
    """
    if not text:
        return ""
    lines = [line for line in text.splitlines() if line.strip() != ""]
    return "\n".join(lines).strip()


def split_into_8_sections(text: str):
    """
    先切出 8 个块。
    舍弃 ### [1] Problem Summary 之前的任何内容。
    返回 dict:
    {
        'ProblemSummary': ...,
        ...
        'DetailedAlgorithmSteps': ...
    }
    """
    first_idx = text.find(FIRST_SECTION_MARK)
    if first_idx == -1:
        raise ValueError(f"未找到 '{FIRST_SECTION_MARK}'")

    text = text[first_idx:]
    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        raise ValueError("未识别到 section 标题")

    parsed = {v: "" for v in SECTION_VARS.values()}

    for i, m in enumerate(matches):
        title = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end]

        if title in SECTION_VARS:
            parsed[SECTION_VARS[title]] = content

    return parsed


def extract_note_from_detailed(detailed_text: str):
    """
    从第八块中剥离第九块 Note。

    规则：
    1. 先对第八块原始内容处理，不提前删空行。
    2. 只考虑“最后一个由空行分隔出的连续非空块”是否应作为 Note。
    3. 若该最后块内部存在某行匹配 '数字. 空格'（如 5. xxx），
       则说明它属于第八块，不剥离。
    4. 若最后块内部没有编号行，且它前面由空行分隔，则将其剥离为 Note。
    5. 因为是按空行分块，若前面有编号步骤但中间已有空行断开，则最后块不再属于该步骤。
    """
    if not detailed_text.strip():
        return "", ""

    lines = detailed_text.splitlines()

    # 去掉末尾空行，便于找最后非空块
    end = len(lines) - 1
    while end >= 0 and lines[end].strip() == "":
        end -= 1

    if end < 0:
        return "", ""

    # 找最后一个连续非空块 [block_start, end]
    block_start = end
    while block_start >= 0 and lines[block_start].strip() != "":
        block_start -= 1
    block_start += 1

    # 如果整个 detailed 都是一个连续非空块，则不剥离 Note
    if block_start == 0:
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    last_block_lines = lines[block_start:end + 1]

    # 若最后块内部含有编号步骤起始行，则它属于第八块
    if any(NUMBERED_STEP_RE.match(line) for line in last_block_lines):
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    # 否则，最后块剥离为 Note
    note_text = "\n".join(last_block_lines)
    main_text = "\n".join(lines[:block_start - 1])  # block_start-1 是分隔空行

    return main_text, note_text


def parse_one_explain_file(txt_path: Path):
    """
    处理单个 explain 文件：
    1. 先切分出 8 个块
    2. 对第 8 块做第 9 块 Note 的剥离
    3. 对这 9 块内容统一做空行剔除
    返回：
    {
        'ProblemSummary': ...,
        ...
        'DetailedAlgorithmSteps': ...,
        'Note': ...
    }
    """
    raw = txt_path.read_text(encoding="utf-8")

    parsed8 = split_into_8_sections(raw)

    detailed_main, note_text = extract_note_from_detailed(parsed8["DetailedAlgorithmSteps"])
    parsed8["DetailedAlgorithmSteps"] = detailed_main

    parsed9 = dict(parsed8)
    parsed9["Note"] = note_text

    # 最后统一对 9 块做空行剔除
    for key in parsed9:
        parsed9[key] = remove_blank_lines(parsed9[key])

    return parsed9


def parse_question_dir(question_dir: Path):
    """
    对单个题目目录处理。
    返回：
    {
        'ProblemSummary': [file1内容, file2内容, ...],
        ...
        'DetailedAlgorithmSteps': [...],
        'Note': [...]
    }
    """
    explain_dir = question_dir / "LLM Explains"
    if not explain_dir.exists() or not explain_dir.is_dir():
        raise FileNotFoundError(f"{question_dir} 下不存在 'LLM Explains' 文件夹")

    txt_files = list_explain_txts(explain_dir)

    result = {v: [] for v in SECTION_VARS.values()}
    result["Note"] = []

    for txt_path in txt_files:
        parsed = parse_one_explain_file(txt_path)
        for key in result:
            result[key].append(parsed[key])

    return result

In [ ]:
# 遍历
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

for d in dirs:
    try:
        all_results[d.name] = parse_question_dir(d)
        print(f"done: {d.name}")
    except Exception as e:
        print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

In [15]:
# 单独测试
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

d = dirs[0]
try:
    all_results[d.name] = parse_question_dir(d)
    print(f"done: {d.name}")
except Exception as e:
    print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

[(1, PosixPath('Questions/Easy B3666/LLM Explains/1 0.0.txt')),
 (2, PosixPath('Questions/Easy B3666/LLM Explains/2 0.0.txt')),
 (3, PosixPath('Questions/Easy B3666/LLM Explains/3 0.0.txt')),
 (4, PosixPath('Questions/Easy B3666/LLM Explains/4 0.2.txt')),
 (5, PosixPath('Questions/Easy B3666/LLM Explains/5 0.2.txt')),
 (6, PosixPath('Questions/Easy B3666/LLM Explains/6 0.2.txt')),
 (7, PosixPath('Questions/Easy B3666/LLM Explains/7 0.5.txt')),
 (8, PosixPath('Questions/Easy B3666/LLM Explains/8 0.5.txt')),
 (9, PosixPath('Questions/Easy B3666/LLM Explains/9 0.5.txt'))]

done: Easy B3666

示例题目: Easy B3666
{'AlgorithmIdea': ['- Use a stack or a similar data structure to keep track of '
                   'the indices of the suffix maximum values.\n'
                   '- When a new element is inserted, update the stack '
                   'accordingly to maintain the correct suffix maximum '
                   'values.\n'
                   '- Calculate the bitwise XOR of the indices of the suffix '
                   'maximum values after each insertion.',
                   '- Use a stack to keep track of the indices of the suffix '
                   'maximum values.\n'
                   '- When a new element is inserted, pop elements from the '
                   'stack that are smaller than the new element and update the '
                   'stack.\n'
                   '- Calculate the bitwise XOR of the indices in the stack.',
                   '- Use a stack to keep track of the indices of the suffix '
                   'maximum values.\n'


In [23]:
display(len(all_results["Easy B3666"]["AlgorithmIdea"]))
display(all_results["Easy B3666"]["AlgorithmIdea"][0])

9

'- Use a stack or a similar data structure to keep track of the indices of the suffix maximum values.\n- When a new element is inserted, update the stack accordingly to maintain the correct suffix maximum values.\n- Calculate the bitwise XOR of the indices of the suffix maximum values after each insertion.'

In [30]:
def build_similarity_matrix(text_list):
    n = len(text_list)
    mat = np.zeros((n, n))

    # 对角线 = 1
    for i in range(n):
        mat[i][i] = 1.0

    # 构造需要计算的 pairs（只算上三角）
    pairs = []
    index_pairs = []

    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((text_list[i], text_list[j]))
            index_pairs.append((i, j))

    if pairs:
        scores = model.predict(pairs)

        # 填充矩阵（对称）
        for (i, j), score in zip(index_pairs, scores):
            mat[i][j] = score
            mat[j][i] = score

    return mat


# ===== 主处理 =====

all_matrices = {}  # 每个题目对应9个矩阵

for qname, qdata in all_results.items():
    mats = {}

    for key, mat_name in zip(KEY_ORDER, MAT_NAMES):
        texts = qdata[key]
        mat = build_similarity_matrix(texts)
        mats[mat_name] = mat

    all_matrices[qname] = mats

In [34]:
# ===== 示例访问 =====
# 某题的 ProblemSummary 矩阵
# all_matrices["Easy B3666"]["mat1PS"]

# 打印一个示例
first_q = next(iter(all_matrices))
print("题目:", first_q)

for name, mat in all_matrices[first_q].items():
    print(f"\n{name} shape={mat.shape}")
    df = pd.DataFrame(mat)
    display(df)

题目: Easy B3666

mat1PS shape=(9, 9)


,0,1,2,3,4,5,6,7,8
0,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193
1,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193
2,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193
3,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.790381,0.974193
4,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.790381,0.974193
5,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.790381,0.974193
6,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.790381,0.974193
7,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,1.000000,0.800534
8,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.800534,1.000000



mat2FPD shape=(9, 9)


,0,1,2,3,4,5,6,7,8
0,1.000000,0.973901,0.973901,0.973901,0.923796,0.973901,0.973901,0.741009,0.860405
1,0.973901,1.000000,0.973901,0.973901,0.923796,0.973901,0.973901,0.741009,0.860405
2,0.973901,0.973901,1.000000,0.973901,0.923796,0.973901,0.973901,0.741009,0.860405
3,0.973901,0.973901,0.973901,1.000000,0.923796,0.973901,0.973901,0.741009,0.860405
4,0.923796,0.923796,0.923796,0.923796,1.000000,0.930113,0.930113,0.746340,0.829398
5,0.973901,0.973901,0.973901,0.973901,0.930113,1.000000,0.973901,0.741009,0.860405
6,0.973901,0.973901,0.973901,0.973901,0.930113,0.973901,1.000000,0.741009,0.860405
7,0.741009,0.741009,0.741009,0.741009,0.746340,0.741009,0.741009,1.000000,0.728801
8,0.860405,0.860405,0.860405,0.860405,0.829398,0.860405,0.860405,0.728801,1.000000



mat3IS shape=(9, 9)


...

,0,1,2,3,4,5,6,7,8
0,1.000000,0.818575,0.818575,0.818575,0.838266,0.970616,0.940095,0.738731,0.818575
1,0.818575,1.000000,0.970888,0.970888,0.845689,0.824118,0.832106,0.752740,0.970888
2,0.818575,0.970888,1.000000,0.970888,0.845689,0.824118,0.832106,0.752740,0.970888
3,0.818575,0.970888,0.970888,1.000000,0.845689,0.824118,0.832106,0.752740,0.970888
4,0.838266,0.845689,0.845689,0.845689,1.000000,0.849063,0.838758,0.763297,0.869233
5,0.970616,0.824118,0.824118,0.824118,0.849063,1.000000,0.940095,0.738731,0.818575
6,0.940095,0.832106,0.832106,0.832106,0.838758,0.940095,1.000000,0.705856,0.833218
7,0.738731,0.752740,0.752740,0.752740,0.763297,0.738731,0.705856,1.000000,0.744162
8,0.818575,0.970888,0.970888,0.970888,0.869233,0.818575,0.833218,0.744162,1.000000



mat8DAS shape=(9, 9)


,0,1,2,3,4,5,6,7,8
0,1.000000,0.769515,0.769515,0.769515,0.720330,0.758055,0.726280,0.720734,0.753445
1,0.769515,1.000000,0.714528,0.714528,0.682391,0.699487,0.690963,0.647481,0.692413
2,0.769515,0.714528,1.000000,0.714528,0.682391,0.699487,0.690963,0.647481,0.692413
3,0.769515,0.714528,0.714528,1.000000,0.682391,0.699487,0.690963,0.647481,0.692413
4,0.720330,0.682391,0.682391,0.682391,1.000000,0.725030,0.694512,0.729835,0.734539
5,0.758055,0.699487,0.699487,0.699487,0.725030,1.000000,0.641544,0.698278,0.640992
6,0.726280,0.690963,0.690963,0.690963,0.694512,0.641544,1.000000,0.682659,0.647417
7,0.720734,0.647481,0.647481,0.647481,0.729835,0.698278,0.682659,1.000000,0.703131
8,0.753445,0.692413,0.692413,0.692413,0.734539,0.640992,0.647417,0.703131,1.000000



mat9N shape=(9, 9)


,0,1,2,3,4,5,6,7,8
0,1.000000,0.411465,0.411465,0.411465,0.696396,0.157573,0.404931,0.564293,0.398996
1,0.411465,1.000000,0.972024,0.972024,0.611300,0.482170,0.960938,0.388596,0.914900
2,0.411465,0.972024,1.000000,0.972024,0.611300,0.482170,0.960938,0.388596,0.914900
3,0.411465,0.972024,0.972024,1.000000,0.611300,0.482170,0.960938,0.388596,0.914900
4,0.696396,0.611300,0.611300,0.611300,1.000000,0.249275,0.554198,0.749070,0.561655
5,0.157573,0.482170,0.482170,0.482170,0.249275,1.000000,0.498340,0.113924,0.427316
6,0.404931,0.960938,0.960938,0.960938,0.554198,0.498340,1.000000,0.375847,0.909659
7,0.564293,0.388596,0.388596,0.388596,0.749070,0.113924,0.375847,1.000000,0.402078
8,0.398996,0.914900,0.914900,0.914900,0.561655,0.427316,0.909659,0.402078,1.000000
